# Neural style transfer

_Deep Learning_

**Keep a photo's content but repaint it in another image's artistic style.**

Neural style transfer mixes two images: the content of a photo and the style of a painting.

     The result keeps what is in your photo but paints it in the brushstrokes and colors of the artwork.

---

This notebook is a practice scaffold for the **Neural style transfer** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns** — each sample is an 8x8 grid of pixel intensities (0–16).

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("image array:", digits.images.shape, " labels:", digits.target.shape)
samples = list(zip(digits.images, digits.target))
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

### Step 1 — Define the Gram matrix (the "style")

Style is captured not by *where* features appear but by *how they correlate*. The Gram matrix multiplies a layer's flattened feature maps by their own transpose, giving a channel-by-channel correlation matrix. Two images that share these correlations look like they share a painterly style, regardless of content. We normalize by the number of elements so the scale stays comparable across layers.

In [ ]:
import torch
import torch.nn.functional as F

def gram(feat):
    # feat: (batch, channels, h, w) -> channel correlation matrix
    b, c, h, w = feat.shape
    f = feat.view(c, h * w)
    return (f @ f.t()) / (c * h * w)   # the "style" of a layer

### Step 2 — Set up the content, style, and the image to optimize

We stand in for real VGG features with random tensors: `content_feat` from the photo, `style_feat` from the painting. The key object is `img` — the feature tensor we will actually optimize, marked `requires_grad=True` so gradients flow into it. Adam is attached to `img` itself (not to any network weights), because in style transfer the *image* is what we train.

In [ ]:
content_feat = torch.randn(1, 16, 8, 8)        # features of the content photo
style_feat = torch.randn(1, 16, 8, 8)          # features of the style painting
img = torch.randn(1, 16, 8, 8, requires_grad=True)   # the image we optimize
opt = torch.optim.Adam([img], lr=0.05)

### Step 3 — Optimize the image to blend content and style

Each step computes two losses: the **content loss** keeps `img` close to the content features, and the **style loss** matches the Gram matrices so `img` picks up the painting's style. We sum them with a large weight (`1e3`) on style to balance the two, then back-propagate into the pixels and step. After a few iterations the style loss has dropped, showing the image is absorbing the target style.

In [ ]:
for step in range(5):
    opt.zero_grad()
    content_loss = F.mse_loss(img, content_feat)             # keep the content
    style_loss = F.mse_loss(gram(img), gram(style_feat))     # match the style
    (content_loss + 1e3 * style_loss).backward()             # blend the two
    opt.step()
print("final style loss:", round(F.mse_loss(gram(img), gram(style_feat)).item(), 5))

## Visualize the data & results

_As we optimize a blank image toward a real digit, does the content loss fall to zero?_

### Step 1 — Set up a target digit and a blank canvas

To isolate just the *content* half of the idea, we pick one real 8x8 digit image as the target and start from a blank (all-zero) canvas. We'll repeatedly nudge the canvas toward the target and watch the content loss — the mean-squared pixel difference — fall.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

digits = load_digits()
target = digits.images[8].astype(float)  # real 8x8 image of an 8
cur = np.zeros_like(target)              # start from a blank canvas

### Step 2 — Step the canvas toward the target and plot the loss

Each iteration records the current content loss, then moves the canvas a fraction (0.4) of the way toward the target — a simple gradient step on the pixels. Plotting the recorded losses shows them dropping smoothly toward zero, the content-matching half of neural style transfer in miniature.

In [ ]:
losses = []
for step in range(16):
    losses.append(((cur - target) ** 2).mean())   # content loss
    cur += 0.4 * (target - cur)                    # gradient step on pixels

plt.plot(losses, color="#c89bff", label="content loss")
plt.title("Content loss while optimizing an image toward a real digit")
plt.xlabel("step"); plt.ylabel("mean-squared loss")
plt.legend()
plt.show()